# Clean-machine check (Google Colab, T4 GPU)

Runs the submission exactly like the organizers: fresh clone, `pip install`, weights download, then
`run_submission.py`. It checks that it installs, that it runs inside the 3x time budget on a T4, and
that the output matches `predictions_samples.json` in the repository.

**Runtime -> Change runtime type -> T4 GPU**, then run all cells. Put one sample video in your Google
Drive first and set `VIDEO` below.

In [ ]:
REPO = "https://github.com/<user>/<repo>.git"     # TODO: your public repository
TAG = "main"                                       # or the submission tag
VIDEO = "/content/drive/MyDrive/C3896.MP4"          # a sample video in your Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
!git clone --depth 1 --branch $TAG $REPO repo
%cd repo
!pip install -q -r requirements.txt
!bash weights/download.sh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/test && cp "$VIDEO" /content/test/

In [ ]:
# the organizers' command (fresh cache, so this includes detection)
!python run_submission.py --videos /content/test --out predictions.json
!python evaluate.py --pred predictions.json --validate-only

In [ ]:
# time budget and reproducibility against the committed predictions_samples.json
import json
new = json.load(open("predictions.json"))
old = json.load(open("predictions_samples.json"))
for name, log in new["log"].items():
    print(f"{name}: {log['total_sec']} s of {log['budget_sec']} s budget ({log['total_sec'] / log['budget_sec']:.0%})",
          "errors:", log["errors"] or "none")
    a, b = new["videos"][name]["events"], old["videos"].get(name, {}).get("events")
    print("  events identical to predictions_samples.json:", a == b, f"({len(a)} vs {len(b) if b is not None else '-'})")